In [1]:
import torch

In [2]:
x = torch.arange(10.0, requires_grad=True)
x

tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.], requires_grad=True)

In [3]:
y = 2 * torch.dot(x, x)
y

tensor(570., grad_fn=<MulBackward0>)

In [4]:
y.backward()
x.grad, 4 * x, torch.allclose(x.grad, 4 * x)

(tensor([ 0.,  4.,  8., 12., 16., 20., 24., 28., 32., 36.]),
 tensor([ 0.,  4.,  8., 12., 16., 20., 24., 28., 32., 36.],
        grad_fn=<MulBackward0>),
 True)

In [5]:
# 默认的梯度累加现象：
y = x.sum()
y.backward()
print("第一次求导后的梯度:", x.grad) # [1, 1, 1, 1]

# 如果不清零再次反向传播：
y = x.sum()
y.backward()
print("未清零再次求导后的梯度:", x.grad) # [2, 2, 2, 2] （累加了！）

# ----------------------------------------------------
# 📌 正确做法：在新一轮求导前使用 zero_() 显式清零
x.grad.zero_()

y = x.sum()
y.backward()
print("清零后再次求导的梯度:", x.grad) # [1, 1, 1, 1] （恢复正常）


第一次求导后的梯度: tensor([ 1.,  5.,  9., 13., 17., 21., 25., 29., 33., 37.])
未清零再次求导后的梯度: tensor([ 2.,  6., 10., 14., 18., 22., 26., 30., 34., 38.])
清零后再次求导的梯度: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [6]:
x.grad.zero_()
y = x * x

In [7]:
y.sum().backward()
x.grad

tensor([ 0.,  2.,  4.,  6.,  8., 10., 12., 14., 16., 18.])

In [9]:
x.grad.zero_()
y = x * x
y.backward(torch.ones_like(y))
x.grad

tensor([ 0.,  2.,  4.,  6.,  8., 10., 12., 14., 16., 18.])

In [12]:
x.grad.zero_()
y = x * x

# u 是从计算图中分离出来的常数张量，它不再带有计算图历史
u = y.detach()
z = u * x

z.sum().backward()
#此时 dz/dx = u = x^2，而不是 3 * x^2
x.grad, u


(tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.]),
 tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.]))

In [13]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True) # 随机标量
d = f(a)
d.backward()

# 验证：f(a) 本质上是 c = k * a，其中 k 是由控制流决定的系数
print("梯度 d.grad == d / a ?", a.grad == d / a) # True


梯度 d.grad == d / a ? tensor(False)
